**Template Clustering Image**

Load gambar → scale + PCA → K-Means → evaluasi → plot → insight.

Digits aktif sebagai contoh. Untuk data soal, nonaktifkan cell Digits dan hapus `#` pada tepat satu loader folder/ZIP/NPZ.
Ganti path dan `k` sesuai soal. Label asli hanya untuk mengecek hasil.

In [ ]:
# Jalankan jika diperlukan:
# %pip install numpy pandas matplotlib scikit-learn pillow tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

**Load data · Digits**

In [ ]:
from sklearn.datasets import load_digits

d = load_digits()
X = d.data
images = d.images
y_true = d.target
print(X.shape)

**Load data · Folder**

Contoh struktur: `data/nama_kelas/gambar.jpg`. Nama subfolder menjadi label pembanding.

In [ ]:
# from pathlib import Path
# from PIL import Image

# data_dir = Path("/content/data/image")
# extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
# paths = sorted(p for p in data_dir.rglob("*") if p.is_file() and p.suffix.lower() in extensions)

# images = []
# for p in paths:
#     with Image.open(p) as img:
#         images.append(np.array(img.convert("RGB").resize((64, 64))))
# images = np.array(images)
# X = images.reshape(len(images), -1).astype(np.float32) / 255.0
# y_true = np.array([p.parent.name for p in paths])
# print(X.shape)

**Load data · ZIP**

Ekstrak ke folder baru. Contoh ZIP berisi subfolder kelas; subfolder ikut dibaca.

In [ ]:
# import zipfile
# from pathlib import Path
# from PIL import Image

# data_dir = Path("/content/image_extracted")
# with zipfile.ZipFile("/content/image.zip") as archive:
#     archive.extractall(data_dir)

# extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
# paths = sorted(p for p in data_dir.rglob("*") if p.is_file() and p.suffix.lower() in extensions)
# images = []
# for p in paths:
#     with Image.open(p) as img:
#         images.append(np.array(img.convert("RGB").resize((64, 64))))
# images = np.array(images)
# X = images.reshape(len(images), -1).astype(np.float32) / 255.0
# y_true = np.array([p.parent.name for p in paths])
# print(X.shape)

**Load data · NPZ**

Sesuai dataset latihan: gambar `x_train` dan label `y_train`. Contoh memakai 5.000 train; ubah `n`. Test tetap terpisah.

In [ ]:
# data = np.load("/content/Data Latihan Datmin II (Image).npz", allow_pickle=False)
# print(data.files)

# n = 5000
# images = data["x_train"][:n]
# X = images.reshape(len(images), -1).astype(np.float32) / 255.0
# y_true = data["y_train"][:n]
# print(X.shape)

**Contoh gambar**

In [ ]:
fig, ax = plt.subplots(2, 8, figsize=(10, 3))
for a in ax.flat:
    a.axis("off")
for a, image in zip(ax.flat, images[:16]):
    a.imshow(image, cmap="gray")
plt.tight_layout()
plt.show()

**Preprocessing**

Scale lalu PCA maksimal 50 komponen. Jumlahnya menyesuaikan jika data/fitur sedikit.

In [ ]:
# Alternatif kalau ada outlier:
# from sklearn.preprocessing import RobustScaler
# X_scaled = RobustScaler().fit_transform(X)

X_scaled = StandardScaler().fit_transform(X)

n_pc = min(50, X_scaled.shape[0] - 1, X_scaled.shape[1])
pca = PCA(n_components=n_pc, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print("Variance explained:", pca.explained_variance_ratio_.sum())

**Tentukan k**

Coba k = 2–14. Silhouette memakai maksimal 3.000 data sebagai perkiraan skor.

In [ ]:
from tqdm.auto import tqdm

ks = range(2, min(15, len(X_pca)))
inertia, sil = [], []
for k in tqdm(ks):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_pca)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X_pca, km.labels_, sample_size=min(3000, len(X_pca)), random_state=42))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(ks, inertia, "o-")
ax[0].set_title("Elbow")
ax[1].plot(ks, sil, "o-")
ax[1].set_title("Silhouette")
plt.tight_layout()
plt.show()

k dipilih = ... karena ... (isi dari grafik).

**K-Means + evaluasi**

Ganti `k`; minimal 2, lebih kecil dari jumlah gambar, dan tidak melebihi jumlah gambar yang berbeda.

In [ ]:
k = 10  # GANTI
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_pca)

silhouette = silhouette_score(X_pca, labels, sample_size=min(3000, len(X_pca)), random_state=42)
dbi = davies_bouldin_score(X_pca, labels)
print("Silhouette    :", silhouette)
print("Davies-Bouldin:", dbi)
print(pd.Series(labels).value_counts().sort_index())

**Plot cluster**

Gunakan dua komponen pertama dari PCA yang sama.

In [ ]:
p2 = X_pca[:, :2]

plt.figure(figsize=(7, 6))
sc = plt.scatter(p2[:, 0], p2[:, 1], c=labels, cmap="tab10", s=12)
plt.legend(*sc.legend_elements(), title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.title("KMeans (PCA 2D)")
plt.tight_layout()
plt.show()

**Insight tiap cluster**

In [ ]:
fig, ax = plt.subplots(1, k, figsize=(2 * k, 2.5))
for c in range(k):
    mean_img = images[labels == c].mean(axis=0).astype(images.dtype)
    ax[c].imshow(mean_img, cmap="gray")
    ax[c].set_title(f"C{c} (n={np.sum(labels == c)})", fontsize=8)
    ax[c].axis("off")
plt.tight_layout()
plt.show()

**Bandingkan label asli (opsional)**

Lewati jika label tidak tersedia atau nama folder bukan kategori.

In [ ]:
summary = pd.crosstab(labels, y_true)
display(summary)

**Contoh gambar tiap cluster**

Ambil maksimal enam gambar; cluster kecil tetap bisa ditampilkan.

In [ ]:
rng = np.random.default_rng(42)
fig, ax = plt.subplots(k, 6, figsize=(9, 1.6 * k))
for a in ax.flat:
    a.axis("off")
for c in range(k):
    members = np.flatnonzero(labels == c)
    idx = rng.choice(members, min(6, len(members)), replace=False)
    for a, i in zip(ax[c], idx):
        a.imshow(images[i], cmap="gray")
    ax[c, 0].set_title(f"C{c}", fontsize=8, loc="left")
plt.tight_layout()
plt.show()

**Kesimpulan**

- Evaluasi: silhouette = ..., DBI = ...
- Cluster 0: ...
- Cluster 1: ...
- Cluster yang campur: ...

**Simpan hasil (opsional)**

In [ ]:
# pd.DataFrame({"cluster": labels}).to_csv("image_hasil.csv", index=False)
# pd.DataFrame({"k": list(ks), "inertia": inertia, "silhouette": sil}).to_csv("image_skor_k.csv", index=False)